# Generated collection worker 0/3

Generated from `05_collect_targeted_verifier_groups.ipynb`; edit the canonical notebook, not this copy.


# 05 — Targeted v4 verifier collection

Collect fixed best-of-8 groups at high-uncertainty states. The development
cohort oversamples failed source rollouts; the separately named prospective
test cohort is assigned before candidate outcomes and is never rebalanced.

## 1. Environment setup

In [ ]:
import os, subprocess, sys
try:
    from google.colab import userdata
    for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    repo_dir = "/content/cs159-sp26"
    gh_pat = userdata.get("GH_PAT")
    repo_url = f"https://{gh_pat}@github.com/ArjunS07/cs159-sp26.git"
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "clone", "--branch", "main", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
except ImportError:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "pnp-vla" else os.getcwd()

package_dir = os.path.join(repo_dir, "pnp-vla")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", package_dir + "[sim,analysis]"], check=True)
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)
import pnp
print("Loaded pnp from:", pnp.__file__)

## 2. Load policy, store, and episode manifests

In [ ]:
import json
from tqdm.auto import tqdm
from pnp import libero_env, libero_pro, models
from pnp.experiments import _prepare_libero_pro_episodes
from pnp.store import SupabaseStore
from pnp.verifier import *

benchmark_dict = libero_env.init_libero_benchmark()
libero_pro.patch_torch_load()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()

suites = ("libero_spatial", "libero_object", "libero_goal", "libero_10")
tasks = [(suite, task) for suite in suites
         for task in range(benchmark_dict[suite]().n_tasks)]
standard = libero_env.build_final_episodes(benchmark_dict, tasks=tasks)
for ep in standard: ep["benchmark"] = "libero"
pro = _prepare_libero_pro_episodes()
for ep in pro: ep["benchmark"] = "libero_pro"
episode_lookup = {
    (e["benchmark"], e["suite"], e["task_idx"],
     e.get("ep_idx", e.get("episode_idx"))): e
    for e in standard + pro
}
print(len(standard), len(pro), len(episode_lookup))

## 3. Build and persist outcome-blind v4 manifests

In [ ]:
DEVELOPMENT_EXPERIMENT = "verifier-clean-pairs-v4-dev"
TEST_EXPERIMENT = "verifier-clean-pairs-v4-test"
DEVELOPMENT_TARGETS = {"libero": 180, "libero_pro": 270}
TEST_TARGETS = {"libero": 60, "libero_pro": 90}
CANDIDATE_COUNT = 8
PREFIX_LENGTH = 10
SHARD_COUNT = 3   # Generated worker count.
SHARD_INDEX = 0   # Generated worker index.
assert 0 <= SHARD_INDEX < SHARD_COUNT

def pages(table, columns, configure, order_by):
    rows=[]; start=0
    while True:
        q = configure(store.client.table(table).select(columns))
        for column in order_by:
            q = q.order(column)
        batch = q.range(start, start+999).execute().data or []; rows += batch
        if len(batch) < 1000: return rows
        start += 1000

source_experiments = (
    "libero-hybrid-schedules-k3-v1",
    "libero-pro-canonical-core-k3-v1",
)
rollouts = []
for experiment in source_experiments:
    rollouts += pages(
        "rollouts",
        "rollout_id,benchmark,suite,task_idx,episode_idx,success",
        lambda q, experiment=experiment: q.eq(
            "experiment", experiment).eq(
            "method", "pnp_uncertainty_only").eq("status", "completed"),
        ("rollout_id",))

rollout_ids = sorted({row["rollout_id"] for row in rollouts})
euler = []
for start in range(0, len(rollout_ids), 100):
    batch_ids = rollout_ids[start:start+100]
    euler += pages(
        "pnp_euler_steps", "rollout_id,chunk_idx,euler_step,u_mean",
        lambda q, ids=batch_ids: q.in_("rollout_id", ids),
        ("rollout_id", "chunk_idx", "euler_step"))

v3_groups = pages(
    "verifier_candidate_groups",
    "candidate_group_id,benchmark,suite,task_idx,episode_idx",
    lambda q: q.eq("experiment", "verifier-clean-pairs-v3"),
    ("candidate_group_id",))
excluded = {
    (row["benchmark"], row["suite"], int(row["task_idx"]), int(row["episode_idx"]))
    for row in v3_groups
}
manifests = build_targeted_manifests(
    rollouts, euler, excluded,
    development_targets=DEVELOPMENT_TARGETS,
    test_targets=TEST_TARGETS,
    development_failure_fraction=.70,
    seed=42,
)
manifest_hashes = {
    cohort: collection_manifest_hash(rows) for cohort, rows in manifests.items()}
manifest_document = {
    "version": 2,
    "candidate_count": CANDIDATE_COUNT,
    "prefix_length": PREFIX_LENGTH,
    "development_experiment": DEVELOPMENT_EXPERIMENT,
    "test_experiment": TEST_EXPERIMENT,
    "development_targets": DEVELOPMENT_TARGETS,
    "test_targets": TEST_TARGETS,
    "excluded_v3_episode_count": len(excluded),
    "hashes": manifest_hashes,
    "manifests": manifests,
}
manifest_bytes = json.dumps(
    manifest_document, sort_keys=True, separators=(",", ":")).encode()
manifest_path = (
    f"verifier_manifests/v4-targeted-"
    f"{manifest_hashes['development']}-{manifest_hashes['test']}.json")
store._upload(manifest_path, manifest_bytes)

full_manifest = [
    {**row, "cohort": cohort,
     "experiment": (DEVELOPMENT_EXPERIMENT if cohort == "development"
                    else TEST_EXPERIMENT)}
    for cohort in ("test", "development")
    for row in manifests[cohort]
]
manifest = full_manifest[SHARD_INDEX::SHARD_COUNT]
print({
    "hashes": manifest_hashes,
    "manifest_path": manifest_path,
    "development": len(manifests["development"]),
    "test": len(manifests["test"]),
    "worker_groups": len(manifest),
    "development_source_failures": sum(
        not row["success"] for row in manifests["development"]),
    "test_source_failures": sum(not row["success"] for row in manifests["test"]),
})

## 4. Dry-run identity, disjointness, and schema checks

In [ ]:
development_identities = {
    (r["benchmark"], r["suite"], r["task_idx"], r["episode_idx"])
    for r in manifests["development"]}
test_identities = {
    (r["benchmark"], r["suite"], r["task_idx"], r["episode_idx"])
    for r in manifests["test"]}
assert development_identities.isdisjoint(test_identities)
assert (development_identities | test_identities).isdisjoint(excluded)
missing = [
    row for row in full_manifest
    if (row["benchmark"], row["suite"], row["task_idx"], row["episode_idx"])
       not in episode_lookup]
assert not missing, missing[:3]
store.client.table("verifier_candidate_groups").select(
    "candidate_group_id").limit(1).execute()
print("v4 manifests, identities, and verifier tables are ready")

## 5. Collect fixed-eight deterministic-replay groups

In [ ]:
experiment_names = (DEVELOPMENT_EXPERIMENT, TEST_EXPERIMENT)
existing_group_rows = pages(
    "verifier_candidate_groups",
    "candidate_group_id,experiment,metadata_json",
    lambda q: q.in_("experiment", experiment_names),
    ("candidate_group_id",))
canonical_by_id = {}
for item in full_manifest:
    gid = candidate_group_id(
        item["benchmark"], item["suite"], item["task_idx"],
        item["episode_idx"], item["chunk_idx"], namespace=item["experiment"])
    canonical_by_id[gid] = item
canonical_ids = set(canonical_by_id)

# Older workers could build overlapping shards because their paginated source
# queries had no stable ordering. Preserve those artifacts under an explicitly
# excluded experiment name instead of deleting them.
noncanonical_rows = [
    row for row in existing_group_rows
    if row["candidate_group_id"] not in canonical_ids]
for row in noncanonical_rows:
    metadata = dict(row.get("metadata_json") or {})
    metadata["quarantined_from_experiment"] = row["experiment"]
    metadata["quarantine_reason"] = "noncanonical_v4_manifest"
    (store.client.table("verifier_candidate_groups")
     .update({
         "experiment": row["experiment"] + "-orphan",
         "metadata_json": metadata,
     }).eq("candidate_group_id", row["candidate_group_id"]).execute())

canonical_rows = [
    row for row in existing_group_rows
    if row["candidate_group_id"] in canonical_ids]
for row in canonical_rows:
    item = canonical_by_id[row["candidate_group_id"]]
    metadata = dict(row.get("metadata_json") or {})
    metadata.update({
        "cohort": item["cohort"],
        "collection_manifest_hash": manifest_hashes[item["cohort"]],
        "collection_manifest_path": manifest_path,
        "source_rollout_id": item["rollout_id"],
        "source_success": bool(item["success"]),
        "source_u_mean": float(item["u_mean"]),
    })
    (store.client.table("verifier_candidate_groups")
     .update({"metadata_json": metadata})
     .eq("candidate_group_id", row["candidate_group_id"]).execute())

existing_group_ids = {row["candidate_group_id"] for row in canonical_rows}
candidate_rows = []
existing_id_list = sorted(existing_group_ids)
for start in range(0, len(existing_id_list), 100):
    ids = existing_id_list[start:start+100]
    candidate_rows += pages(
        "verifier_candidates", "candidate_id,candidate_group_id",
        lambda q, ids=ids: q.in_("candidate_group_id", ids),
        ("candidate_id",))
candidate_counts = {}
for row in candidate_rows:
    gid = row["candidate_group_id"]
    candidate_counts[gid] = candidate_counts.get(gid, 0) + 1
existing = {
    gid for gid in existing_group_ids
    if candidate_counts.get(gid, 0) == CANDIDATE_COUNT}
print({
    "quarantined_noncanonical_groups": len(noncanonical_rows),
    "complete_existing_groups": len(existing),
    "partial_groups_to_repair": len(existing_group_ids - existing),
})

store.start_run(
    "verifier_pair_collection", "libero+libero_pro", "verifier-clean-pairs-v4",
    config={
        "groups": len(full_manifest),
        "outcomes": len(full_manifest) * CANDIDATE_COUNT,
        "candidate_count": CANDIDATE_COUNT,
        "prefix_length": PREFIX_LENGTH,
        "manifest_hashes": manifest_hashes,
        "manifest_path": manifest_path,
        "shard_count": SHARD_COUNT,
        "shard_index": SHARD_INDEX,
    })
completed_outcomes = skipped = 0
for item in tqdm(manifest, desc="v4 candidate groups"):
    expected_id = candidate_group_id(
        item["benchmark"], item["suite"], item["task_idx"],
        item["episode_idx"], item["chunk_idx"], namespace=item["experiment"])
    if expected_id in existing:
        continue
    ep = episode_lookup[(
        item["benchmark"], item["suite"], item["task_idx"], item["episode_idx"])]
    env = libero_env.make_env(ep["bddl_path"])
    try:
        try:
            collected = collect_replay_candidate_group(
                env, ep, policy, preprocess, postprocess, device,
                chunk_idx=item["chunk_idx"], uncertainty_stratum="high",
                prefix_length=PREFIX_LENGTH, candidate_count=CANDIDATE_COUNT,
                experiment=item["experiment"])
        except Exception as error:
            print("v4 group skipped:", type(error).__name__, error)
            collected = None
        if collected is None:
            skipped += 1
            continue
        group, candidates = collected
        group["metadata_json"].update({
            "cohort": item["cohort"],
            "collection_manifest_hash": manifest_hashes[item["cohort"]],
            "collection_manifest_path": manifest_path,
            "source_rollout_id": item["rollout_id"],
            "source_success": bool(item["success"]),
            "source_u_mean": float(item["u_mean"]),
        })
        store.register_candidate_group(group, candidates)
        existing.add(group["candidate_group_id"])
        completed_outcomes += len(candidates)
    finally:
        env.close()
store.finish_run(n_rollouts=completed_outcomes)
print({
    "new_outcomes": completed_outcomes,
    "skipped_unreachable": skipped,
    "complete_groups_seen": len(existing),
})

## 6. Cohort integrity and balance report

In [ ]:
for cohort, experiment in (
    ("development", DEVELOPMENT_EXPERIMENT),
    ("test", TEST_EXPERIMENT),
):
    groups = pages(
        "verifier_candidate_groups", "*",
        lambda q, experiment=experiment: q.eq("experiment", experiment),
        ("candidate_group_id",))
    group_ids = {group["candidate_group_id"] for group in groups}
    candidates = []
    group_id_list = sorted(group_ids)
    for start in range(0, len(group_id_list), 100):
        ids = group_id_list[start:start+100]
        candidates += pages(
            "verifier_candidates",
            "candidate_id,candidate_group_id,candidate_kind,success",
            lambda q, ids=ids: q.in_("candidate_group_id", ids),
            ("candidate_id",))
    by_group = {}
    for candidate in candidates:
        by_group.setdefault(candidate["candidate_group_id"], []).append(candidate)
    complete = {
        gid for gid in group_ids
        if len(by_group.get(gid, [])) == CANDIDATE_COUNT}
    expected_hash = manifest_hashes[cohort]
    hash_matches = sum(
        (group.get("metadata_json") or {}).get("collection_manifest_hash")
        == expected_hash for group in groups)
    print(cohort, {
        "manifest_target": len(manifests[cohort]),
        "groups": len(groups),
        "complete_groups": len(complete),
        "partial_groups": len(group_ids - complete),
        "manifest_hash_matches": hash_matches,
        "successes": sum(candidate["success"] for candidate in candidates),
        "failures": sum(not candidate["success"] for candidate in candidates),
        "discordant_groups": sum(
            len({candidate["success"] for candidate in by_group.get(gid, [])}) == 2
            for gid in complete),
        "pairwise_comparisons": sum(
            sum(candidate["success"] for candidate in by_group.get(gid, []))
            * sum(not candidate["success"] for candidate in by_group.get(gid, []))
            for gid in complete),
        "source_failures": sum(
            not bool((group.get("metadata_json") or {}).get("source_success"))
            for group in groups),
    })